# Chapter 10 Companion Notebook: Algorithmic Trading and Market Microstructure

This notebook reproduces every worked numerical example from Chapter 10 of *AI in Finance* (`chapter10.tex`): the moving-average crossover backtest, TWAP/VWAP execution, Kyle's lambda, the Avellaneda-Stoikov inventory model, the pairs-trading backtest, the multi-stock statistical arbitrage example, backtest performance metrics, and implementation shortfall.

## 1. Moving-average crossover strategy (Section 10.3)

In [1]:
import numpy as np
import pandas as pd

prices = pd.Series([100,101,103,106,110,115,119,118,114,109,104,100], name='Price')
returns = prices.pct_change()

ma_short = prices.rolling(3).mean()
ma_long = prices.rolling(6).mean()
signal = (ma_short > ma_long).astype(int)
signal_lag = signal.shift(1).fillna(0)

strategy_returns = (signal_lag * returns).fillna(0)

table = pd.DataFrame({
    'Price': prices, 'MA3': ma_short.round(2), 'MA6': ma_long.round(2),
    'Signal': signal, 'Return': returns.round(4), 'StratReturn': strategy_returns.round(4),
})
print(table)

cum_strategy = (1 + strategy_returns).cumprod()
cum_buyhold = (1 + returns.fillna(0)).cumprod()
print(f"\nFinal strategy cumulative return: {cum_strategy.iloc[-1]-1:.4%}")
print(f"Final buy-and-hold cumulative return: {cum_buyhold.iloc[-1]-1:.4%}")

    Price     MA3     MA6  Signal  Return  StratReturn
0     100     NaN     NaN       0     NaN       0.0000
1     101     NaN     NaN       0  0.0100       0.0000
2     103  101.33     NaN       0  0.0198       0.0000
3     106  103.33     NaN       0  0.0291       0.0000
4     110  106.33     NaN       0  0.0377       0.0000
5     115  110.33  105.83       1  0.0455       0.0000
6     119  114.67  109.00       1  0.0348       0.0348
7     118  117.33  111.83       1 -0.0084      -0.0084
8     114  117.00  113.67       1 -0.0339      -0.0339
9     109  113.67  114.17       0 -0.0439      -0.0439
10    104  109.00  113.17       0 -0.0459      -0.0000
11    100  104.33  110.67       0 -0.0385      -0.0000

Final strategy cumulative return: -5.2174%
Final buy-and-hold cumulative return: 0.0000%


## 1b. A feature-engineering catalog for trading signals (Section 10.14.1, Table 10.7)

Reuses the identical twelve-day price series from the moving-average crossover example above.

In [2]:
prices_arr = prices.to_numpy()
rets_arr = prices_arr[1:] / prices_arr[:-1] - 1     # rets_arr[i] realized on day i+2
changes = np.diff(prices_arr)                        # changes[i] realized on day i+2
mom3 = prices_arr[3:] / prices_arr[:-3] - 1           # mom3[i] available as of day i+4

rows = []
for day in range(4, 13):
    p = prices_arr[day - 1]
    m = mom3[day - 4] * 100
    vol_window = rets_arr[day - 4:day - 1]
    v = vol_window.std(ddof=0) * 100
    ch_window = changes[day - 4:day - 1]
    gains = ch_window[ch_window > 0].sum()
    losses = -ch_window[ch_window < 0].sum()
    avg_gain, avg_loss = gains / 3, losses / 3
    rsi = 100.0 if avg_loss == 0 else 100 - 100 / (1 + avg_gain / avg_loss)
    rows.append([day, p, round(m, 2), round(v, 3), round(rsi, 2)])

feat_table = pd.DataFrame(rows, columns=['Day', 'Price', 'Mom3(%)', 'Vol3(%)', 'RSI3'])
print(feat_table.to_string(index=False))

 Day  Price  Mom3(%)  Vol3(%)   RSI3
   4    106     6.00    0.781 100.00
   5    110     8.91    0.732 100.00
   6    115    11.65    0.667 100.00
   7    119    12.26    0.450 100.00
   8    118     7.27    2.328  90.00
   9    114    -0.87    2.835  44.44
  10    109    -8.40    1.493   0.00
  11    104   -11.86    0.523   0.00
  12    100   -12.28    0.313   0.00


## 1c. Absolute Price Oscillator and Bollinger Bands (Section 10.14.1, Tables 10.8-10.9)

Reuses the identical twelve-day price series, with EMA/SMA windows scaled down from industry-standard lengths to fit the short illustrative series.

In [3]:
import math
import statistics as stats

# Absolute Price Oscillator: fast=3, slow=6 EMA (scaled down from the industry-standard 10/40)
n_fast, n_slow = 3, 6
lam_fast, lam_slow = 2.0/(n_fast+1), 2.0/(n_slow+1)
ema_fast = ema_slow = None
apo_rows = []
for day, p in enumerate(prices_arr, start=1):
    if ema_fast is None:
        ema_fast = ema_slow = p
    else:
        ema_fast = lam_fast*p + (1-lam_fast)*ema_fast
        ema_slow = lam_slow*p + (1-lam_slow)*ema_slow
    apo_rows.append([day, p, round(ema_fast,3), round(ema_slow,3), round(ema_fast-ema_slow,3)])

apo_table = pd.DataFrame(apo_rows, columns=['Day','Price','EMA3','EMA6','APO'])
print(apo_table.to_string(index=False))

 Day  Price    EMA3    EMA6    APO
   1    100 100.000 100.000  0.000
   2    101 100.500 100.286  0.214
   3    103 101.750 101.061  0.689
   4    106 103.875 102.472  1.403
   5    110 106.938 104.623  2.314
   6    115 110.969 107.588  3.381
   7    119 114.984 110.849  4.136
   8    118 116.492 112.892  3.600
   9    114 115.246 113.208  2.038
  10    109 112.123 112.006  0.117
  11    104 108.062 109.719 -1.657
  12    100 104.031 106.942 -2.911


In [4]:
# Bollinger Bands: 5-day SMA, k=2 (scaled down from the industry-standard 20-day)
# note: statistics.mean() silently truncates to numpy's integer dtype when given
# numpy.int64 values (unlike Python's built-in int), so prices are cast to float first
num_periods, stdev_factor = 5, 2.0
history = []
bb_rows = []
for day, p in enumerate(prices_arr, start=1):
    history.append(float(p))
    if len(history) > num_periods:
        del history[0]
    sma = stats.mean(history)
    variance = sum((h-sma)**2 for h in history)
    stdev = math.sqrt(variance/len(history))
    bb_rows.append([day, p, round(sma,3), round(sma+stdev_factor*stdev,3), round(sma-stdev_factor*stdev,3)])

bb_table = pd.DataFrame(bb_rows, columns=['Day','Price','Middle(SMA5)','Upper','Lower'])
print(bb_table.to_string(index=False))

 Day  Price  Middle(SMA5)   Upper   Lower
   1    100       100.000 100.000 100.000
   2    101       100.500 101.500  99.500
   3    103       101.333 103.828  98.839
   4    106       102.500 107.083  97.917
   5    110       104.000 111.266  96.734
   6    115       107.000 117.040  96.960
   7    119       110.600 122.234  98.966
   8    118       113.600 123.447 103.753
   9    114       115.200 121.575 108.825
  10    109       115.000 122.043 107.957
  11    104       112.800 124.071 101.529
  12    100       109.000 122.023  95.977


## 2. Order book walk and TWAP/VWAP (Section 10.5-10.7)

In [5]:
order_book_asks = pd.DataFrame({'Price': [50.10, 50.15, 50.20], 'Size': [100, 200, 300]}, index=[1,2,3])
best_ask = 50.10
order_size = 250
filled, cost = 0, 0.0
for price, size in zip(order_book_asks['Price'], order_book_asks['Size']):
    take = min(size, order_size - filled)
    cost += take * price
    filled += take
    if filled >= order_size:
        break
avg_price = cost / order_size
print(f"Aggressive execution avg price: ${avg_price:.2f}, impact: {(avg_price-best_ask)/best_ask*10000:.1f} bps")
print(f"Patient (TWAP-style) execution avg price: ${best_ask:.2f}, impact: 0.0 bps")

# TWAP/VWAP on Table 10.2
twap_prices = np.array([50.00, 50.05, 50.10, 50.08, 50.12])
twap_volumes = np.array([10_000, 12_000, 8_000, 15_000, 9_000])
twap_price = twap_prices.mean()
vwap_price = np.sum(twap_prices * twap_volumes) / np.sum(twap_volumes)
print(f"\nTWAP execution price: ${twap_price:.4f}")
print(f"VWAP benchmark: ${vwap_price:.4f}")
print(f"Difference: {(twap_price-vwap_price)/vwap_price*10000:.2f} bps")

Aggressive execution avg price: $50.13, impact: 6.0 bps
Patient (TWAP-style) execution avg price: $50.10, impact: 0.0 bps

TWAP execution price: $50.0700
VWAP benchmark: $50.0681
Difference: 0.37 bps


## 3. Kyle's lambda (Section 10.8, Table 10.3)

In [6]:
order_flow = np.array([5, -3, 8, -6, 2, 4], dtype=float)
price_change = np.array([10, -7, 15, -11, 3, 9], dtype=float)

xbar, ybar = order_flow.mean(), price_change.mean()
lam = np.sum((order_flow - xbar) * (price_change - ybar)) / np.sum((order_flow - xbar) ** 2)
intercept = ybar - lam * xbar
fitted = intercept + lam * order_flow
r2 = 1 - np.sum((price_change - fitted) ** 2) / np.sum((price_change - ybar) ** 2)

print(f"Kyle's lambda: {lam:.4f} cents per thousand shares")
print(f"Intercept: {intercept:.4f}")
print(f"R^2: {r2:.4f}")

Kyle's lambda: 1.9466 cents per thousand shares
Intercept: -0.0777
R^2: 0.9915


## 4. Avellaneda-Stoikov inventory model (Section 10.9)

In [7]:
s, q, gamma, sigma, T_t, kappa = 100.0, 5, 0.1, 2.0, 1.0, 1.5

reservation_price = s - q * gamma * sigma**2 * T_t
as_spread = gamma * sigma**2 * T_t + (2/gamma) * np.log(1 + gamma/kappa)
bid = reservation_price - as_spread/2
ask = reservation_price + as_spread/2

sym_bid = s - as_spread/2
sym_ask = s + as_spread/2

print(f"Reservation price: ${reservation_price:.2f}")
print(f"Optimal spread: ${as_spread:.4f}")
print(f"Inventory-aware bid/ask: ${bid:.2f} / ${ask:.2f}")
print(f"Symmetric (no-inventory) bid/ask: ${sym_bid:.2f} / ${sym_ask:.2f}")
print(f"Skew: ${s - reservation_price:.2f}")

Reservation price: $98.00
Optimal spread: $1.6908
Inventory-aware bid/ask: $97.15 / $98.85
Symmetric (no-inventory) bid/ask: $99.15 / $100.85
Skew: $2.00


## 5. Pairs-trading backtest (Section 10.10, Table 10.4)

In [8]:
A = np.array([50,51,53,52,54,55,58,56.5,54.8,54.5])
B = np.array([48,49,50,50,51,52,54,53,52,52])
spread = A - B
print(f"Spread: {spread}")

mu_in = spread[:6].mean()
sigma_in = spread[:6].std(ddof=1)
z = (spread - mu_in) / sigma_in
print(f"In-sample mean={mu_in:.2f}, std={sigma_in:.4f}")
print(f"Out-of-sample z-scores (days 7-10): {z[6:].round(2)}")

entry_idx, exit_idx = 6, 9
pnl = spread[entry_idx] - spread[exit_idx]
capital = A[entry_idx] + B[entry_idx]
print(f"\nEntry spread: {spread[entry_idx]:.2f}, Exit spread: {spread[exit_idx]:.2f}")
print(f"P&L per share-pair: ${pnl:.2f}")
print(f"Return on capital: {pnl/capital:.4%}")

Spread: [2.  2.  3.  2.  3.  3.  4.  3.5 2.8 2.5]
In-sample mean=2.50, std=0.5477
Out-of-sample z-scores (days 7-10): [2.74 1.83 0.55 0.  ]

Entry spread: 4.00, Exit spread: 2.50
P&L per share-pair: $1.50
Return on capital: 1.3393%


## 6. Statistical arbitrage beyond pairs (Section 10.11, Table 10.5)

In [9]:
market = np.array([-2.0,-1.0,0.0,1.0,2.0,3.0])
beta_W, beta_X, beta_Y = 1.0, 1.2, 0.8
resid_W = np.array([0.10,-0.10,0.00,0.10,-0.10, 1.20])
resid_X = np.array([0.00, 0.10,-0.10,0.00, 0.10,-0.10])
resid_Y = np.array([-0.10,0.00, 0.10,-0.10,0.00, 0.10])

W = beta_W*market + resid_W
X = beta_X*market + resid_X
Y = beta_Y*market + resid_Y

def estimate_beta(m, r):
    mbar, rbar = m.mean(), r.mean()
    b = np.sum((m-mbar)*(r-rbar))/np.sum((m-mbar)**2)
    a = rbar - b*mbar
    return a, b

for name, series in [('W', W), ('X', X), ('Y', Y)]:
    m_in, r_in = market[:5], series[:5]
    a, b = estimate_beta(m_in, r_in)
    resid_in = r_in - (a + b*m_in)
    resid_std = resid_in.std(ddof=1)
    resid_6 = series[5] - (a + b*market[5])
    z6 = (resid_6 - resid_in.mean())/resid_std
    print(f"{name}: beta={b:.2f}, month-6 residual={resid_6:.2f}%, z-score={z6:.1f}")

W: beta=0.98, month-6 residual=1.26%, z-score=13.3
X: beta=1.21, month-6 residual=-0.15%, z-score=-1.8
Y: beta=0.81, month-6 residual=0.09%, z-score=1.1


## 7. Backtest performance metrics (Section 10.12)

In [10]:
strat_returns_full = pd.Series([np.nan,0.0,0.0,0.0,0.0,0.0,0.0348,-0.0084,-0.0339,-0.0439,-0.0,-0.0]).fillna(0)
cum = (1 + strat_returns_full).cumprod()
running_max = cum.cummax()
drawdown = cum / running_max - 1
max_dd = drawdown.min()

nonzero_days = strat_returns_full[strat_returns_full != 0]
hit_rate = (nonzero_days > 0).mean()

mean_r, std_r = strat_returns_full.mean(), strat_returns_full.std(ddof=1)
sharpe = mean_r / std_r

print(f"Max drawdown: {max_dd:.4%}")
print(f"Hit rate (active days): {hit_rate:.0%}")
print(f"Sharpe ratio: {sharpe:.2f}")

Max drawdown: -8.4071%
Hit rate (active days): 25%
Sharpe ratio: -0.22


## 8. Implementation shortfall (Section 10.13)

In [11]:
decision_price = 50.00
execution_price = 50.07
shortfall_per_share = execution_price - decision_price
shortfall_bps = shortfall_per_share / decision_price * 10_000
order_size = 1000
total_shortfall = shortfall_per_share * order_size

print(f"Implementation shortfall: ${shortfall_per_share:.2f}/share = {shortfall_bps:.0f} bps")
print(f"Total shortfall on {order_size}-share order: ${total_shortfall:.0f}")

Implementation shortfall: $0.07/share = 14 bps
Total shortfall on 1000-share order: $70


## Exercises (match Chapter 10, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [12]:
# Exercise 3: 500-share order, 200 shares at $30.00, rest at $30.05
book_ex3 = [(30.00, 200), (30.05, 400)]
size_ex3, filled_ex3, cost_ex3 = 500, 0, 0.0
for price, size in book_ex3:
    take = min(size, size_ex3 - filled_ex3)
    cost_ex3 += take * price
    filled_ex3 += take
avg_ex3 = cost_ex3 / size_ex3
print(f"Exercise 3 -- avg price=${avg_ex3:.4f}, impact={(avg_ex3-30.00)/30.00*10000:.2f} bps")

# Exercise 4: TWAP/VWAP for a 2,000-share order (same table, ratios unchanged)
print(f"Exercise 4 -- TWAP=${twap_price:.4f}, VWAP=${vwap_price:.4f} (same regardless of order size)")

# Exercise 9: Avellaneda-Stoikov with q=-8 (short position)
q_ex9 = -8
r_ex9 = s - q_ex9*gamma*sigma**2*T_t
bid_ex9 = r_ex9 - as_spread/2
ask_ex9 = r_ex9 + as_spread/2
print(f"Exercise 9 -- reservation price=${r_ex9:.2f}, bid=${bid_ex9:.2f}, ask=${ask_ex9:.2f}")

Exercise 3 -- avg price=$30.0300, impact=10.00 bps
Exercise 4 -- TWAP=$50.0700, VWAP=$50.0681 (same regardless of order size)
Exercise 9 -- reservation price=$103.20, bid=$102.35, ask=$104.05
